# Transformers — COVID Tweet Classification with BERT Clean Solution

Fine-tune BERT for binary classification (COVID-related vs not).

**Key concepts:** AutoTokenizer, BertForSequenceClassification, custom Dataset, AdamW, low learning rate fine-tuning.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertForSequenceClassification, AutoTokenizer, AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 1–2: Load & Split

In [ ]:
data = pd.read_csv("covid_twitter_dataset_codebasics_DL.csv")
print(data.head())

train_X, test_X, train_Y, test_Y = train_test_split(
    data['text'], data['target'], test_size=0.3, random_state=42
)

## Step 3: Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
train_tokens = tokenizer(list(train_X), padding=True, truncation=True, max_length=128, return_tensors=None)
test_tokens = tokenizer(list(test_X), padding=True, truncation=True, max_length=128, return_tensors=None)

## Step 4–5: Custom Dataset & DataLoader

In [ ]:
class TokenData(Dataset):
    def __init__(self, tokens, labels):
        self.tokens = tokens
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sample = {k: torch.tensor(v[idx]) for k, v in self.tokens.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample

train_dataset = TokenData(train_tokens, train_Y)
test_dataset = TokenData(test_tokens, test_Y)

train_loader = DataLoader(train_dataset, batch_size=30, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=30, shuffle=False)

## Step 6: Model, Optimizer, Loss

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2).to(device)
optimizer = AdamW(bert_model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss()

## Step 7: Training

In [ ]:
num_epochs = 3
for epoch in range(num_epochs):
    bert_model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = bert_model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        loss = loss_fn(outputs.logits, batch['labels'])
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {total_train_loss/len(train_loader):.4f}")

## Step 8: Evaluation

In [ ]:
bert_model.eval()
correct, total, total_test_loss = 0, 0, 0.0
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = bert_model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        loss = loss_fn(outputs.logits, batch['labels'])
        total_test_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == batch['labels']).sum().item()
        total += batch['labels'].size(0)

print(f"Test Loss: {total_test_loss/len(test_loader):.4f}, Accuracy: {100*correct/total:.2f}%")

## Step 9: Inference on New Tweets

In [ ]:
test_tweets = [
    "The number of COVID-19 cases has increased in the past week.",
    "Just watched an amazing sunset at the beach!",
    "New vaccination centers are opening up to combat the spread of coronavirus.",
    "A virus has impacted the global ecosystem a lot",
    "India lost the match against Australia at Adelaide"
]

def predict_tweet(tweet_text):
    tokens = tokenizer(tweet_text, padding=True, truncation=True, return_tensors='pt')
    tokens = {k: v.to(device) for k, v in tokens.items()}
    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**tokens)
        probs = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probs, dim=1)
    return {
        'text': tweet_text,
        'is_covid': bool(prediction.item()),
        'confidence': probs[0][prediction[0]].item() * 100
    }

for tweet in test_tweets:
    result = predict_tweet(tweet)
    label = 'COVID-related' if result['is_covid'] else 'Not COVID-related'
    print(f"Tweet: {result['text']}")
    print(f"Prediction: {label} ({result['confidence']:.2f}%)\n")